In [1]:
import pandas as pd

In [2]:
credit_df = pd.read_csv(r"C:\Data\data\raw\creditcard.csv")

In [3]:
credit_df.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='str')

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
#credit_df = pd.read_csv(r"C:\Data\data\processed\Creditcard_processed.csv")

# Separate features and target
X_credit = credit_df.drop('Class', axis=1)
y_credit = credit_df['Class']

# Stratified train-test split
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_credit,
    y_credit,
    test_size=0.2,
    random_state=42,
    stratify=y_credit
)

print("Training shape:", Xc_train.shape)
print("Testing shape:", Xc_test.shape)

print("\nTraining class distribution:")
print(yc_train.value_counts(normalize=True))

print("\nTesting class distribution:")
print(yc_test.value_counts(normalize=True))

Training shape: (227845, 30)
Testing shape: (56962, 30)

Training class distribution:
Class
0    0.998271
1    0.001729
Name: proportion, dtype: float64

Testing class distribution:
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64


In [4]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=0.1,
    random_state=42
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    Xc_train,
    yc_train
)

print(y_train_resampled.value_counts())

Class
0    227451
1     22745
Name: count, dtype: int64


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
fraud_df = pd.read_csv(r"C:\Data\data\raw\Fraud_Data.csv")

# Separate features and target
X_fraud = fraud_df.drop('class', axis=1)
y_fraud = fraud_df['class']

# Stratified train-test split
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fraud,
    y_fraud,
    test_size=0.2,
    random_state=42,
    stratify=y_fraud
)

print("Training set shape:", Xf_train.shape)
print("Testing set shape:", Xf_test.shape)

print("\nClass distribution in training set:")
print(yf_train.value_counts(normalize=True))

print("\nClass distribution in testing set:")
print(yf_test.value_counts(normalize=True))

Training set shape: (120889, 10)
Testing set shape: (30223, 10)

Class distribution in training set:
class
0    0.906352
1    0.093648
Name: proportion, dtype: float64

Class distribution in testing set:
class
0    0.906363
1    0.093637
Name: proportion, dtype: float64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import numpy as np

# Identify feature types
numeric_features = Xc_train.select_dtypes(include=np.number).columns
categorical_features = Xc_train.select_dtypes(include='object').columns

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]),
            numeric_features
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]),
            categorical_features
        )
    ]
)

In [7]:
X_train_processed = preprocessor.fit_transform(
    Xc_train
)

X_test_processed = preprocessor.transform(
    Xc_test
)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, f1_score

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train_resampled, y_train_resampled)

y_pred = lr.predict( Xc_test)
y_prob = lr.predict_proba( Xc_test)[:,1]

print(confusion_matrix(yc_test, y_pred))
print(classification_report(yc_test, y_pred))
print("F1:", f1_score(yc_test, y_pred))
print("AUC-PR:", average_precision_score(yc_test, y_prob))

C:\Users\Soret\fraud-detection\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[[56761   103]
 [   12    86]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.46      0.88      0.60        98

    accuracy                           1.00     56962
   macro avg       0.73      0.94      0.80     56962
weighted avg       1.00      1.00      1.00     56962

F1: 0.5993031358885017
AUC-PR: 0.7183821461764233


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    f1_score
)

# Train model
lr = LogisticRegression(max_iter=3000, solver='lbfgs')

lr.fit(X_train_resampled, y_train_resampled)

# Predict on SAME feature space used for training pipeline
y_pred = lr.predict(X_test_processed)
y_prob = lr.predict_proba(X_test_processed)[:, 1]

# Evaluation (use consistent test labels)
print(confusion_matrix(yc_test, y_pred))
print(classification_report(yc_test, y_pred))

print("F1:", f1_score(yc_test, y_pred))
print("AUC-PR:", average_precision_score(yc_test, y_prob))

C:\Users\Soret\fraud-detection\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 3000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=3000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[[56791    73]
 [   15    83]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.53      0.85      0.65        98

    accuracy                           1.00     56962
   macro avg       0.77      0.92      0.83     56962
weighted avg       1.00      1.00      1.00     56962

F1: 0.6535433070866141
AUC-PR: 0.7006892072905198


C:\Users\Soret\fraud-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
C:\Users\Soret\fraud-detection\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_resampled, y_train_resampled)

rf_pred = rf.predict(Xc_test)
rf_prob = rf.predict_proba(Xc_test)[:,1]

print(confusion_matrix(yc_test, rf_pred))
print(classification_report(yc_test, rf_pred))
print("F1:", f1_score(yc_test, rf_pred))
print("AUC-PR:", average_precision_score(yc_test, rf_prob))

[[56849    15]
 [   12    86]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.85      0.88      0.86        98

    accuracy                           1.00     56962
   macro avg       0.93      0.94      0.93     56962
weighted avg       1.00      1.00      1.00     56962

F1: 0.864321608040201
AUC-PR: 0.8796312703595159


In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    average_precision_score
)

from imblearn.over_sampling import SMOTE

In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

In [13]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rf,
    X_train_resampled,
    y_train_resampled,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)

print("Mean F1:", scores.mean())
print("Std F1:", scores.std())

Mean F1: 0.9602673590584512
Std F1: 0.002695181766188117


In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_resampled, y_train_resampled)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [16]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    'n_estimators': [100],
    'max_depth': [10]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(
    X_train_resampled,
    y_train_resampled
)

print("Best Parameters:")
print(grid_search.best_params_)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
Best Parameters:
{'max_depth': 10, 'n_estimators': 100}


In [21]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7]
}

grid_search = GridSearchCV(
    xgb,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_resampled, y_train_resampled)

best_xgb = grid_search.best_estimator_

y_pred = best_xgb.predict(Xc_test)
y_prob = best_xgb.predict_proba(Xc_test)[:, 1]

print("Best Parameters:", grid_search.best_params_)
print("Accuracy :", accuracy_score(yc_test, y_pred))
print("Precision:", precision_score(yc_test, y_pred))
print("Recall   :", recall_score(yc_test, y_pred))
print("F1 Score :", f1_score(yc_test, y_pred))
print("ROC-AUC  :", roc_auc_score(yc_test, y_prob))

Best Parameters: {'max_depth': 5, 'n_estimators': 300}
Accuracy : 0.9995435553526912
Precision: 0.8829787234042553
Recall   : 0.8469387755102041
F1 Score : 0.8645833333333334
ROC-AUC  : 0.984420130953338


In [22]:
best_rf = grid_search.best_estimator_

In [23]:
rf_pred = best_rf.predict(X_test_processed)

rf_prob = best_rf.predict_proba(
    X_test_processed
)[:, 1]

In [24]:
print("Confusion Matrix")
print(confusion_matrix(yc_test, rf_pred))

print("\nClassification Report")
print(classification_report(yc_test, rf_pred))

rf_f1 = f1_score(yc_test, rf_pred)

rf_aucpr = average_precision_score(
    yc_test,
    rf_prob
)

print("\nF1 Score:", rf_f1)
print("AUC-PR:", rf_aucpr)

Confusion Matrix
[[56845    19]
 [   20    78]]

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.80      0.80      0.80        98

    accuracy                           1.00     56962
   macro avg       0.90      0.90      0.90     56962
weighted avg       1.00      1.00      1.00     56962


F1 Score: 0.8
AUC-PR: 0.8204324886626643


In [25]:
print("Random Forest Results")
print("F1 Score :", rf_f1)
print("AUC-PR   :", rf_aucpr)

Random Forest Results
F1 Score : 0.8
AUC-PR   : 0.8204324886626643


In [29]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV

lgbm = LGBMClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, -1]
}

grid_search = GridSearchCV(
    lgbm,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train_resampled, y_train_resampled)

best_lgbm = grid_search.best_estimator_

y_pred = best_lgbm.predict(Xc_test)
y_prob = best_lgbm.predict_proba(Xc_test)[:, 1]

print("Best Parameters:", grid_search.best_params_)
print("Accuracy :", accuracy_score(yc_test, y_pred))
print("Precision:", precision_score(yc_test, y_pred))
print("Recall   :", recall_score(yc_test, y_pred))
print("F1 Score :", f1_score(yc_test, y_pred))
print("ROC-AUC  :", roc_auc_score(yc_test, y_prob))

[LightGBM] [Info] Number of positive: 22745, number of negative: 227451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.052720 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 250196, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.090909 -> initscore=-2.302589
[LightGBM] [Info] Start training from score -2.302589
Best Parameters: {'max_depth': 7, 'n_estimators': 300}
Accuracy : 0.9995259997893332
Precision: 0.8817204301075269
Recall   : 0.8367346938775511
F1 Score : 0.8586387434554974
ROC-AUC  : 0.9827641031088856


In [30]:
from sklearn.model_selection import cross_validate, StratifiedKFold
import pandas as pd

def evaluate_model(model, X, y):
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scoring = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc'
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=skf,
        scoring=scoring,
        n_jobs=-1
    )

    return {
        metric: f"{scores[f'test_{metric}'].mean():.4f} ± {scores[f'test_{metric}'].std():.4f}"
        for metric in scoring.keys()
    }

In [ ]:
rf_results = evaluate_model(best_rf, X_train_resampled, y_train_resampled)
xgb_results = evaluate_model(best_xgb, X_train_resampled, y_train_resampled)
lgbm_results = evaluate_model(best_lgbm, X_train_resampled, y_train_resampled)

print(rf_results)